In [ ]:
! pip install google-adk --quiet --upgrade

In [ ]:
# @title Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

In [ ]:
# @title Setup Some Variables and a Staging Bucket
import vertexai
import hashlib
from google.cloud import storage
from google.api_core import exceptions

# PROJECT_ID retrieved from the Current Project
PROJECT_ID = ! gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-east4"
MODEL = "gemini-2.5-flash"

# Create a short, unique hash from the Project ID
# We use SHA-256 and grab the first 4 characters for brevity
id_hash = hashlib.sha256(PROJECT_ID.encode()).hexdigest()[:8]
bucket_name = f"agent-staging-{id_hash}"

print(f"Target Bucket Name: {bucket_name}")

# Initialize the Storage Client
storage_client = storage.Client(project=PROJECT_ID)

# Check if the bucket exists and create it if it doesn't
try:
    bucket = storage_client.get_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' already exists. Skipping creation.")
except exceptions.NotFound:
    print(f"Bucket '{bucket_name}' not found. Creating...")
    bucket = storage_client.create_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' created successfully.")
except Exception as e:
    print(f"An error occurred: {e}")


STAGING_BUCKET=f"gs://{bucket_name}"

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Staging Bucket: {STAGING_BUCKET}")

In [ ]:
# @title Initial Vertex AI
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

In [ ]:
# @title Download the OpenAPI Specification and save to a file.

import requests, json

spec = requests.get("https://cymbal-travel-270259231695.us-central1.run.app/openapi.json").json()

with open("openapi.json", "w") as f:
    json.dump(spec, f, indent=2)

In [ ]:
# @title Create the Cymbal Travel Booking Agent.

# This agent has access to the Cymbal Travel API. If can search for resorts, cars, and flights.
# It caan help users book travel.

import json
from pathlib import Path
from google.adk.agents import Agent
from google.adk.tools.openapi_tool.openapi_spec_parser.openapi_toolset import OpenAPIToolset

with open("openapi.json") as f:
     openapi_spec_dict = json.load(f)

# This reads the OpenAPI JSON Spec.
toolset = OpenAPIToolset(spec_dict=openapi_spec_dict)
api_tools = await toolset.get_tools()

# --- Agent ---
travel_booking_agent = Agent(
    name="Travel_Agent",
    model=MODEL,
    tools=api_tools,
    instruction="""
You are a travel agent for Cymbal Travel Resorts.
You have access to an API that can be used to search for hotels (resorts), flights, and car rentals.

NOTE: This is a mock API used in a training course. There is no real data and
the API functions will return results even if parameters are incorrect or missing.

Important rules:
- Study the OpenAPI spec you are given carefully.
- DO NOT hallucinate endpoints, parameters, or request formats.
- The API uses the word "hotels", but the user may say "resorts" or similar.

Scope:
- You only handle requests by calling the API tools.

Helpful endpoints (verify paths/params in the spec):
- Flights:   GET /api/flights/search
- Hotels:    GET /api/hotels/search, GET /api/hotels/top
- Cars:      GET /api/cars/search
- Cart:      POST /api/cart/add, POST /api/cart/remove, GET /api/cart/<user_id>, POST /api/cart/checkout
- Users:     POST /api/users, POST /api/login
"""
)

In [ ]:
# @title Create an app to use the Agent Locally
from vertexai import agent_engines

app = agent_engines.AdkApp(
    agent=travel_booking_agent,
)

user_id = "test-user"
session = app.create_session(user_id=user_id)

print(f"Session ID: {session.id}")

In [ ]:
# @title See if the Deployed Agent Works.

from IPython.display import Markdown, display

prompts = [
    "What resorts do you have in Italy?",
    "What are my car choices for March 30, 2028. at Cymbal Resort Amalfi Coast?",
    "What are your top resorts?",
    "I want to purchase a new Computer, can you help me?",
]

for prompt in prompts:
  async for event in app.async_stream_query(
      user_id=user_id,
      session_id=session.id,
      message=prompt,
  ):
      lastevent = event

  if 'content' in lastevent:
      display(Markdown(lastevent["content"]["parts"][0]["text"]))
  else:
      display(lastevent)

In [ ]:
# @title Deploy to Agent Engine (or Update if it exists)

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

DISPLAY_NAME = "Cymbal_Travel_Agent_v2"
DESCRIPTION = "Provides Customer Service Questions for Cymbal Travel"
REQUIREMENTS = ["google-adk"]
ENV_VARS = {
  "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
  "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
}

CONFIG= {
    "display_name": DISPLAY_NAME,
    "description": DESCRIPTION,
    "requirements": REQUIREMENTS,
    "staging_bucket": STAGING_BUCKET,
    "min_instances": 1,
    "max_instances": 3,
    "env_vars": ENV_VARS,
}

RESOURCE_NAME = None
for agent in client.agent_engines.list():
  if agent.api_resource.display_name == DISPLAY_NAME:
    RESOURCE_NAME = agent.api_resource.name
    break


if RESOURCE_NAME:
  print(f"Agent exists, going to update...")
  remote_agent = client.agent_engines.update(
    name=RESOURCE_NAME,
    agent=app,
    config=CONFIG
  )
else:
  print(f"Going to create agent...")
  remote_agent = client.agent_engines.create(
      agent=app,
      config=CONFIG
  )

RESOURCE_NAME = remote_agent.api_resource.name
print(f"Agent Engine Resource Name: {RESOURCE_NAME}")

In [ ]:
# @title List currently deployed Agents

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

for agent in client.agent_engines.list():
    print(agent.api_resource.name)
    print(agent.api_resource.display_name)

In [ ]:
# @title Connect to the Deployed Agent

RESOURCE_NAME = None
for agent in client.agent_engines.list():
  if agent.api_resource.display_name == DISPLAY_NAME:
    RESOURCE_NAME = agent.api_resource.name
    break

print(f"Connect to agent: {RESOURCE_NAME}")
remote_agent = client.agent_engines.get(
    name=RESOURCE_NAME
)

In [ ]:
# @title Just try a command and see if it works

prompts = [
    "What resorts do you have in Italy?",
    "What are my car choices for March 30, 2028. at Cymbal Resort Amalfi Coast?",
    "What are your top resorts?",
    "I want to purchase a new Computer, can you help me?",
]

for prompt in prompts:
  async for event in remote_agent.async_stream_query(
      user_id="Doug",
      message=prompt,
  ):
      print(event)
      lastevent = event

  if 'content' in lastevent:
      display(Markdown(lastevent["content"]["parts"][0]["text"]))
  else:
      display(lastevent)